<a href="https://colab.research.google.com/github/fwilhelmy/QB-PIER/blob/main/src/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Google Colab (Skip locally)

In [5]:
#@title Mount your Google Drive
# If you run this notebook locally or on a cluster (i.e. not on Google Colab)
# you can delete this cell which is specific to Google Colab. You may also
# change the paths for data/logs in Arguments below.
%matplotlib inline
%load_ext autoreload
%autoreload 2

from google.colab import drive
drive.mount('/content/gdrive')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Mounted at /content/gdrive


## Link source code

In [ ]:
#@title Link via google drive
#@markdown Enter the path to the assignment folder in your Google Drive
# If you run this notebook locally or on a cluster (i.e. not on Google Colab)
# you can delete this cell which is specific to Google Colab. You may also
# change the paths for data/logs in Arguments below.
import sys
import os
import shutil
import warnings

folder = "/content/gdrive/MyDrive/IFT6135/HW2_2025/src" #@param {type:"string"}
!ln -Ts "$folder" /content/src 2> /dev/null

# Add the assignment folder to Python path
if '/content/src' not in sys.path:
  sys.path.insert(0, '/content/src')

# Check if CUDA is available
import torch
if not torch.cuda.is_available():
  warnings.warn('CUDA is not available.')

In [ ]:
#@title Link via zip upload
from google.colab import files
import sys
import os
import shutil
import warnings

# 1️⃣ Upload your ZIP containing only your Python code (it may contain a top-level 'src' folder)
uploaded = files.upload()  # choose your zip
if not uploaded:
    raise FileNotFoundError("No ZIP file uploaded.")
zip_name = next(iter(uploaded.keys()))
print(f"✔ Uploaded: {zip_name}")

# 2️⃣ Prepare /content/src
base_dir = "/content/src"
if os.path.exists(base_dir):
    shutil.rmtree(base_dir)
os.makedirs(base_dir, exist_ok=True)

# 3️⃣ Extract into /content/src
!unzip -q "{zip_name}" -d "{base_dir}"
print("✔ Extracted into /content/src/")

# 4️⃣ If there's a single top-level folder inside (e.g. /content/src/src/), flatten it
entries = os.listdir(base_dir)
if len(entries) == 1 and os.path.isdir(os.path.join(base_dir, entries[0])):
    nested = os.path.join(base_dir, entries[0])
    for fname in os.listdir(nested):
        shutil.move(os.path.join(nested, fname), base_dir)
    shutil.rmtree(nested)
    print(f"✔ Flattened nested folder `{entries[0]}` into /content/src/")

# 5️⃣ Add /content/src to Python path for top-level imports
if base_dir not in sys.path:
    sys.path.insert(0, base_dir)
    print("✔ Added '/content/src' to sys.path")

# 6️⃣ (Optional) CUDA check
try:
    import torch
    if not torch.cuda.is_available():
        print("⚠️ CUDA not available.")
except ImportError:
    pass


## Setting up virtual environment

In [7]:
!pip install virtualenv
!virtualenv /content/drive/MyDrive/virtual_env

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 30.2 MB/s eta 0:00:00
created virtual environment CPython3.11.13.final.0-64 in 558ms
  creator CPython3Posix(dest=/content/drive/MyDrive/virtual_env, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, setuptools=bundle, via=copy, app_data_dir=/root/.local/share/virtualenv)
    added seed packages: pip==25.1.1, setuptools==80.3.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator


In [ ]:
!source /content/drive/MyDrive/virtual_env/bin/activate; pip install -r requirements.txt

# Imports & Setup

In [2]:
%matplotlib inline
import warnings

from main import train_model, run_attacks
from shutil import make_archive

from metrics import LoggingService
from metrics import MetricsContainer
from models import BaseModel
from models.hybrid import HQNN, AmpVQC
from arguments import Arguments

from visualisation.TrainingVisualisation import generate_gradient_flow_plots, plot_training_curves, plot_models_together

ModuleNotFoundError: No module named 'lightning'

In [ ]:
model_name = "HQNN"  #@param {type:"string"}
model_cls = AmpVQC  #@param {type:"string"}
#path = f"C:\\Users\\fwilh\\Desktop\\QMLBenchmark\\logs\\test_models\\2025-06-27@14-35-01\\{model_name}\\{model_name}_1\\0"
path = f"C:\\Users\\fwilh\\Desktop\\QMLBenchmark\\logs\\test_models\\2025-06-27@19-00-06\\{model_name}\\{model_name}_1\\0\\final.ckpt"
path = "C:\\Users\\fwilh\\Desktop\\QMLBenchmark\\logs\\test_models\\2025-06-30@14-43-55\\AmpVQC\\AmpVQC_1\\0\\final.ckpt"

# load the model
model = model_cls.load_from_checkpoint(path, map_location='cpu')

In [ ]:
#plot_training_result(cont, 42, 42)
#generate_gradient_flow_plots(cont)

# Experiment #1

## Experiment Setup

In [ ]:
# Define a arguments and set global defaults
LoggingService("test_models")
seeds = [0, 42]
args = Arguments()
args.verbose = True
args.n_steps = 2000
args.num_workers = 0

if getattr(args, "num_threads", None) is not None:
    torch.set_num_threads(args.num_threads)

args.eval_ratio = 0.4
args.train_ratio = 0.6

args.train_batch_size = 2**8
args.eval_batch_size = 2**11

args.train_log_interval = 0.5
args.eval_log_interval = 0.5

# 2) Define your list of model‐configurations: (name, class, kwargs)
model_specs = [
    # ("ResNet18_1", ResNet18, {
    #     "input_size": (1, 28),
    #     "num_classes": 10,
    # }),
    # ("MLPMixer_1", MLPMixer, {
    #     "num_classes": 10,
    #     "input_size": (1, 28),
    #     "patch_size": 4,
    #     "embed_dim": 256,
    #     "num_blocks": 4,
    #     "drop_rate": 0.0,
    #     "activation": "gelu",
    #     "mlp_ratio": (0.5, 4.0),
    # }),
    # ("MLP_1", MLP, {
    #     "input_size": (1, 28),
    #     "hidden_sizes": [1024, 512, 64, 64],
    #     "num_classes": 10,
    #     "activation": "relu",
    # }),
    # ("ConvNet_1", ConvNet, {
    #     "input_size": (1, 28),
    #     "num_classes": 10,
    # }),
    # ("LeNet_1", LeNet, {
    #     "in_channels": 1,
    #     "classes": 10,
    # }),
    # ("HQNN_Parallel_1", HQNN, {
    #     "in_channels": 1,
    #     "classes": 10,
    # }),
    # ("HQuanv_1", HQuanv, {
    #     "in_channels": 1,
    #     "classes": 10,
    #     "img_size": 28,
    # }),
    # ("HQE_1", HQE, {
    #     "input_size": 28 * 28,
    #     "hidden_sizes": [256],
    #     "classes": 10,
    # }),
    # ("DimVQC_1", DimVQC, {
    #     "input_size": 28 * 28,
    #     "reduction_method": "pca",
    #     "reduced_dim": 4,
    #     "n_qdepth": 1,
    # }),
    ("AmpVQC_depth1", AmpVQC, {
        "input_size": 28 * 28,
        "n_qdepth": 1,
    }),
    ("AmpVQC_depth2", AmpVQC, {
        "input_size": 28 * 28,
        "n_qdepth": 2,
    }),
    ("AmpVQC_depth3", AmpVQC, {
        "input_size": 28 * 28,
        "n_qdepth": 3,
    }),
    ("AmpVQC_depth4", AmpVQC, {
        "input_size": 28 * 28,
        "n_qdepth": 4,
    }),
]

## Models Training

In [ ]:
trained_models = {}
for (model_name, model_cls, config) in model_specs:
    args.exp_name = model_name
    args.model = model_cls
    args.model_config = config

    trained_models[model_name] = train_model(args, seeds)
    plot_training_curves(model_name, trained_models[model_name])
    generate_gradient_flow_plots(list(trained_models[model_name].values())[0])

plot_models_together(trained_models)

## Attacks

In [ ]:
# Run transfer attacks
# attacks_metrics = run_attacks(trained_models, args)
# plot_transfer_attack_effects(attacks_metrics,
#                             title_prefix="transfer")
# from pprint import pprint
# pprint(trained_models['HQNN_Parallel_1'][0].all_metrics, depth=2)
# metrics = trained_models['HQNN_Parallel_1'][0].all_metrics

## Final

In [ ]:
make_archive('experiment1', 'zip', 'logs/experiment1')